# Импорты

In [ ]:
import os
import subprocess
import re
import nltk
from pymystem3 import Mystem
from nltk.tokenize import sent_tokenize
import json
#unoconv для преобразований типов файлов
!apt-get update
!apt-get install -y unoconv
# nltk для токенизации
!pip install pymystem3 nltk

nltk.download('punkt_tab')

# Создание директорий для файлов

In [41]:
doc_folder_path = '/content/' # папка, где находятся .doc файлы
output_txt_folder = '/content/drive/MyDrive/ConvertedTxtFiles' # папка для конвертированных .txt файлов
cleaned_txt_folder = '/content/drive/MyDrive/CleanedTxtFiles' # папка для очищенных .txt файлов

if not os.path.exists(output_txt_folder):
    os.makedirs(output_txt_folder)
    print(f"Создана папка для конвертированных файлов: {output_txt_folder}")
else:
    print(f"Папка для конвертированных файлов уже существует: {output_txt_folder}")

if not os.path.exists(cleaned_txt_folder):
    os.makedirs(cleaned_txt_folder)
    print(f"Создана папка для очищенных файлов: {cleaned_txt_folder}")
else:
    print(f"Папка для очищенных файлов уже существует: {cleaned_txt_folder}")

Папка для конвертированных файлов уже существует: /content/drive/MyDrive/ConvertedTxtFiles
Папка для очищенных файлов уже существует: /content/drive/MyDrive/CleanedTxtFiles


#Конвертация файлов .doc в .txt

In [42]:
doc_files = [f for f in os.listdir(doc_folder_path) if f.endswith('.doc')]

if not doc_files:
    print(f"В папке '{doc_folder_path}' файлы .doc не найдены.")
else:
    print(f"Найдено {len(doc_files)} файлов")
    for doc_file in doc_files:
        input_path = os.path.join(doc_folder_path, doc_file)
        output_filename = os.path.splitext(doc_file)[0] + '.txt'
        output_path = os.path.join(output_txt_folder, output_filename)

        command = [
            'unoconv',
            '-f', 'txt',
            '-o', output_path,
            input_path
        ]
        try:
            subprocess.run(command, check=True, capture_output=True, text=True)
            print(f"Успешно конвертирован '{doc_file}' в '{output_filename}'.")
        except subprocess.CalledProcessError as e:
            print(f"Ошибка при конвертации '{doc_file}': {e.stderr}")
        except FileNotFoundError:
            print("Ошибка: 'unoconv' команда не найдена. Убедитесь, что unoconv установлен.")

print(f"Файлы в '{output_txt_folder}'.")

Найдено 4 файлов
Успешно конвертирован 'Бесы.doc' в 'Бесы.txt'.
Успешно конвертирован 'Преступление и наказание.doc' в 'Преступление и наказание.txt'.
Успешно конвертирован 'Идиот.doc' в 'Идиот.txt'.
Успешно конвертирован 'Братья Карамазовы.doc' в 'Братья Карамазовы.txt'.
Файлы в '/content/drive/MyDrive/ConvertedTxtFiles'.


# Предварительная очистка текста от ненужных слов

In [43]:
converted_txt_files = [f for f in os.listdir(output_txt_folder) if f.endswith('.txt')]

for txt_file_name in converted_txt_files:
    input_file_path = os.path.join(output_txt_folder, txt_file_name)
    output_file_path = os.path.join(cleaned_txt_folder, txt_file_name)

    try:
        with open(input_file_path, 'r', encoding='utf-8') as f_in:
            text = f_in.read()
        text = re.sub(r'\n\s*\n+', '\n\n', text) #удаление нескольких пустых строк, заменяя их одной
        text = re.sub(r'^\s*\d+\s*$\n', '', text, flags=re.MULTILINE) #удаление строк, содержащих только цифры
        non_narrative_keywords = [
            r'ОГЛАВЛЕНИЕ', r'СОДЕРЖАНИЕ', r'ПРЕДИСЛОВИЕ', r'ПОСЛЕСЛОВИЕ',
            r'ПРИМЕЧАНИЯ', r'БИОГРАФИЯ', r'ЧАСТЬ I', r'ЧАСТЬ II', r'ГЛАВА I', r'ГЛАВА II',
            r'ЭПИЛОГ', r'ПРИЛОЖЕНИЕ'
        ]
        non_narrative_pattern = '|'.join([f'\b{kw}\b' for kw in non_narrative_keywords])

        lines = text.split('\n')
        cleaned_lines = []
        in_non_narrative_section = False

        for line in lines:
            stripped_line = line.strip()
            is_all_caps = stripped_line.isupper() and len(stripped_line) > 1 and not stripped_line.isdigit()

            if re.search(non_narrative_pattern, stripped_line, re.IGNORECASE) or is_all_caps:
                if re.search(r'ОГЛАВЛЕНИЕ|СОДЕРЖАНИЕ|ПРЕДИСЛОВИЕ|ПОСЛЕСЛОВИЕ|БИОГРАФИЯ', stripped_line, re.IGNORECASE):
                    in_non_narrative_section = True
                continue
            elif in_non_narrative_section and (not stripped_line or re.search(r'\bГЛАВА\b|\bЧАСТЬ\b', stripped_line, re.IGNORECASE)):
                in_non_narrative_section = False
                cleaned_lines.append(line)
            elif not in_non_narrative_section:
                cleaned_lines.append(line)

        text = '\n'.join(cleaned_lines)
        text = '\n'.join([line.strip() for line in text.split('\n')])
        text = re.sub(r'\n\s*\n+', '\n\n', text)
        text = text.strip()

        with open(output_file_path, 'w', encoding='utf-8') as f_out:
            f_out.write(text)
        print(f"Успешно очищен и сохранен '{txt_file_name}' в '{cleaned_txt_folder}'.")

    except Exception as e:
        print(f"Ошибка обработки '{txt_file_name}': {e}")

Успешно очищен и сохранен 'Преступление и наказание.txt' в '/content/drive/MyDrive/CleanedTxtFiles'.
Успешно очищен и сохранен 'Идиот.txt' в '/content/drive/MyDrive/CleanedTxtFiles'.
Успешно очищен и сохранен 'Бесы.txt' в '/content/drive/MyDrive/CleanedTxtFiles'.
Успешно очищен и сохранен 'Братья Карамазовы.txt' в '/content/drive/MyDrive/CleanedTxtFiles'.


# Токенизация

In [47]:
m = Mystem()

def split_into_sentences_russian(text):
    return sent_tokenize(text, language='russian')

all_text_fragments = []

for txt_file_name in converted_txt_files: # используем те же файлы, что и для очистки
    input_file_path = os.path.join(cleaned_txt_folder, txt_file_name)

    try:
        with open(input_file_path, 'r', encoding='utf-8') as f_in:
            text_content = f_in.read()

        sentences = split_into_sentences_russian(text_content)

        i = 0
        while i < len(sentences):
            current_fragment_size = min(6, len(sentences) - i)
            if len(sentences) - i < 3 and len(sentences) - i > 0:
                if len(all_text_fragments) > 0:
                    last_fragment = all_text_fragments.pop()
                    fragment = ' '.join([last_fragment] + sentences[i:])
                else:
                    fragment = ' '.join(sentences[i:])
                current_fragment_size = len(sentences) - i # Обновляем размер для корректного инкремента
            else:
                fragment = ' '.join(sentences[i : i + current_fragment_size])

            # Добавляем фрагмент, только если он не пустой после объединения
            if fragment.strip():
                all_text_fragments.append(fragment)

            i += current_fragment_size

        print(f"Обработан '{txt_file_name}'. Создано {len(sentences)} предложений.")

    except Exception as e:
        print(f"Ошибка обработки '{txt_file_name}': {e}")

Обработан 'Преступление и наказание.txt'. Создано 13057 предложений.
Обработан 'Идиот.txt'. Создано 13844 предложений.
Обработан 'Бесы.txt'. Создано 14199 предложений.
Обработан 'Братья Карамазовы.txt'. Создано 19214 предложений.


# Небольшая фильтрация

In [48]:
filtered_fragments = []
for fragment in all_text_fragments:
    word_count = len(fragment.split())
    if word_count >= 5:
        filtered_fragments.append(fragment)

all_text_fragments = filtered_fragments

# Финальное сохранение

In [49]:
data_to_save = []
for fragment in all_text_fragments:
    data_to_save.append({
        "original_fragment": fragment
        # "paraphrased_fragment": "" #Для добавления перефраза после генерации, второй шаг
    })

json_output_path = os.path.join(cleaned_txt_folder, 'dostoevsky_original_fragments1.json')

with open(json_output_path, 'w', encoding='utf-8') as f:
    json.dump(data_to_save, f, ensure_ascii=False, indent=4)